# Inter-Annotator Agreement (IAA)

This notebook documents the full IAA workflow in labeled steps:

1. Step 1: Load files and define annotator mapping.
2. Step 2: Build a merged table.
3. Step 3: Normalize labels to POSITIVE/NEGATIVE and inspect distributions.
4. Step 4: Compute Fleiss' kappa.


## Step 1 - Load Data and Define Annotators

Read the three annotator files and define the abbreviation-to-name mapping.

In [52]:
from pathlib import Path
import numpy as np
import pandas as pd
from collections import Counter

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

DATA_DIR = Path('../../data')

annotator_name_map = {
    'ac': 'Alderick',
    'dcch': 'Dennis',
    'zx': 'Zi Xiang',
}

files = {
    'ac': DATA_DIR / 'mcu_manual_label_1000_rating_priority_AC.xlsx',
    'dcch': DATA_DIR / 'mcu_manual_label_1000_rating_priority_DCCH.csv',
    'zx': DATA_DIR / 'mcu_manual_label_1000_rating_priority_ZX.xlsx',
}

def read_any(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    return pd.read_csv(path)

def drop_optional_raw_cols(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_remove = {'notes'}
    keep_cols = [c for c in df.columns if c.lower() not in cols_to_remove]
    return df[keep_cols].copy()

ac = drop_optional_raw_cols(read_any(files['ac']))
dcch = drop_optional_raw_cols(read_any(files['dcch']))
zx = drop_optional_raw_cols(read_any(files['zx']))

print('Loaded raw shapes (after dropping optional columns):')
print(f"AC ({annotator_name_map['ac']})      :", ac.shape)
print(f"DCCH ({annotator_name_map['dcch']})  :", dcch.shape)
print(f"ZX ({annotator_name_map['zx']})      :", zx.shape)

Loaded raw shapes (after dropping optional columns):
AC (Alderick)      : (1000, 9)
DCCH (Dennis)  : (1000, 9)
ZX (Zi Xiang)      : (1000, 9)


## Step 2 - Build Merged Table

Project only the required columns (`review_id`, `movie_id`, `polarity_label`) and merge the three annotator datasets.

In [53]:
merge_keys = [k for k in ['review_id', 'movie_id'] if k in ac.columns and k in dcch.columns and k in zx.columns]
if not merge_keys:
    raise ValueError('No common merge keys found among annotator files.')

def pick_label_cols(df: pd.DataFrame, who: str) -> pd.DataFrame:
    cols = merge_keys.copy()
    subj_col = 'subjective_label'
    pol_col = 'polarity_label'
    if subj_col in df.columns:
        cols.append(subj_col)
    if pol_col in df.columns:
        cols.append(pol_col)
    out = df[cols].copy()
    # If subjective_label is 'objective', set polarity_label to 'neutral'
    if subj_col in out.columns and pol_col in out.columns:
        out[pol_col] = out.apply(
            lambda row: 'neutral' if str(row[subj_col]).strip().lower() == 'objective' else row[pol_col],
            axis=1
        )
    rename_map = {}
    if subj_col in out.columns:
        rename_map[subj_col] = f'subjective_label_{who}'
    if pol_col in out.columns:
        rename_map[pol_col] = f'polarity_label_{who}'
    return out.rename(columns=rename_map)

ac_small = pick_label_cols(ac, 'ac')
dcch_small = pick_label_cols(dcch, 'dcch')
zx_small = pick_label_cols(zx, 'zx')

combined = ac_small.merge(dcch_small, on=merge_keys, how='inner').merge(zx_small, on=merge_keys, how='inner')

print('Merged rows:', len(combined))
print('Columns:', list(combined.columns))
display(combined.head(5))

Merged rows: 1000
Columns: ['review_id', 'movie_id', 'subjective_label_ac', 'polarity_label_ac', 'subjective_label_dcch', 'polarity_label_dcch', 'subjective_label_zx', 'polarity_label_zx']


,review_id,movie_id,subjective_label_ac,polarity_label_ac,subjective_label_dcch,polarity_label_dcch,subjective_label_zx,polarity_label_zx
0,rw3391899,tt0478970,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE
1,rw4836840,tt0478970,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE
2,rw3278832,tt0478970,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE
3,rw4581027,tt5095030,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE
4,rw4372367,tt5095030,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE,SUBJECTIVE,NEGATIVE


## Step 3 - Normalize Polarity Labels

Normalize labels to the binary set: `POSITIVE` and `NEGATIVE`, then inspect class distributions per annotator.

In [54]:
def normalize_polarity(x):
    if pd.isna(x):
        return np.nan
    t = str(x).strip().upper()
    mapping = {
        'POSITIVE': 'POSITIVE',
        'NEGATIVE': 'NEGATIVE',
        'NEUTRAL': 'NEUTRAL',
    }
    return mapping.get(t, np.nan)

def normalize_subjectivity(x):
    if pd.isna(x):
        return np.nan
    t = str(x).strip().upper()
    mapping = {
        'SUBJECTIVE': 'SUBJECTIVE',
        'OBJECTIVE': 'NEUTRAL',
    }
    return mapping.get(t, np.nan)

subjectivity_cols = [c for c in combined.columns if c.startswith('subjective_label_')]
polarity_cols = [c for c in combined.columns if c.startswith('polarity_label_')]
col_to_annotator = {c: f"{c.split('_')[-1].upper()} ({annotator_name_map[c.split('_')[-1]]})" for c in subjectivity_cols + polarity_cols}

for c in subjectivity_cols:
    combined[c] = combined[c].map(normalize_subjectivity)
for c in polarity_cols:
    combined[c] = combined[c].map(normalize_polarity)

print('Subjectivity distribution by annotator (after normalization):')
for c in subjectivity_cols:
    print(f"\n{col_to_annotator[c]}")
    print(combined[c].value_counts(dropna=False))

print('Polarity distribution by annotator (after normalization):')
for c in polarity_cols:
    print(f"\n{col_to_annotator[c]}")
    print(combined[c].value_counts(dropna=False))

display(combined[merge_keys + subjectivity_cols + polarity_cols].head(10))

Subjectivity distribution by annotator (after normalization):

AC (Alderick)
subjective_label_ac
SUBJECTIVE    1000
Name: count, dtype: int64

DCCH (Dennis)
subjective_label_dcch
SUBJECTIVE    1000
Name: count, dtype: int64

ZX (Zi Xiang)
subjective_label_zx
SUBJECTIVE    1000
Name: count, dtype: int64
Polarity distribution by annotator (after normalization):

AC (Alderick)
polarity_label_ac
NEGATIVE    558
POSITIVE    442
Name: count, dtype: int64

DCCH (Dennis)
polarity_label_dcch
NEGATIVE    582
POSITIVE    418
Name: count, dtype: int64

ZX (Zi Xiang)
polarity_label_zx
NEGATIVE    576
POSITIVE    424
Name: count, dtype: int64


,review_id,movie_id,subjective_label_ac,subjective_label_dcch,subjective_label_zx,polarity_label_ac,polarity_label_dcch,polarity_label_zx
0,rw3391899,tt0478970,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
1,rw4836840,tt0478970,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
2,rw3278832,tt0478970,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
3,rw4581027,tt5095030,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
4,rw4372367,tt5095030,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
5,rw4240903,tt5095030,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
6,rw9086849,tt10954600,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
7,rw9002962,tt10954600,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
8,rw9061238,tt10954600,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE
9,rw3227781,tt2395427,SUBJECTIVE,SUBJECTIVE,SUBJECTIVE,NEGATIVE,NEGATIVE,NEGATIVE


## Step 4 - Compute Fleiss' Kappa

Compute inter-annotator agreement for polarity using Fleiss' kappa with binary categories.

In [55]:
def fleiss_kappa_from_ratings(df: pd.DataFrame, annotator_cols, categories):
    # Keep only items where all annotators provided a valid label.
    sub = df[annotator_cols].copy()
    sub = sub[sub.notna().all(axis=1)]

    if sub.empty:
        raise ValueError('No complete rows available for Fleiss kappa calculation.')

    cat_to_idx = {c: i for i, c in enumerate(categories)}
    n_items = len(sub)
    n_annotators = len(annotator_cols)

    counts = np.zeros((n_items, len(categories)), dtype=int)
    valid_rows = []

    for i, (_, row) in enumerate(sub.iterrows()):
        ok = True
        for col in annotator_cols:
            label = row[col]
            if label not in cat_to_idx:
                ok = False
                break
            counts[i, cat_to_idx[label]] += 1
        valid_rows.append(ok)

    counts = counts[np.array(valid_rows)]
    N = counts.shape[0]
    n = n_annotators

    if N == 0:
        raise ValueError('No rows with labels in the expected categories.')

    p_j = counts.sum(axis=0) / (N * n)
    P_i = ((counts * counts).sum(axis=1) - n) / (n * (n - 1))
    P_bar = P_i.mean()
    P_e = (p_j * p_j).sum()

    if np.isclose(1 - P_e, 0):
        kappa = np.nan
    else:
        kappa = (P_bar - P_e) / (1 - P_e)

    return {
        'rows_used': int(N),
        'annotators': int(n),
        'categories': categories,
        'category_proportions': {categories[i]: float(p_j[i]) for i in range(len(categories))},
        'mean_observed_agreement': float(P_bar),
        'expected_agreement': float(P_e),
        'fleiss_kappa': float(kappa),
    }

# Polarity agreement
polarity_result = fleiss_kappa_from_ratings(
    combined,
    annotator_cols=polarity_cols,
    categories=['POSITIVE', 'NEGATIVE']
)
print('=== Polarity Inter-Annotator Agreement Result ===')
for k, v in polarity_result.items():
    print(f'{k}: {v}')

# Subjectivity agreement (if available)
if subjectivity_cols:
    subjectivity_result = fleiss_kappa_from_ratings(
        combined,
        annotator_cols=subjectivity_cols,
        categories=['SUBJECTIVE', 'NEUTRAL']
    )
    print('\n=== Subjectivity Inter-Annotator Agreement Result ===')
    for k, v in subjectivity_result.items():
        print(f'{k}: {v}')

threshold = 0.80
kappa_value = polarity_result['fleiss_kappa']
status = 'PASS' if kappa_value >= threshold else 'BELOW 0.80'
print('\n=== Agreement Threshold Check (Polarity) ===')
print('Criterion: fleiss_kappa >= 0.80')
print(f'Computed fleiss_kappa: {kappa_value:}')
print(f'Threshold Check: {status}')

if subjectivity_cols:
    kappa_value_subj = subjectivity_result['fleiss_kappa']
    import math
    if math.isnan(kappa_value_subj):
        print('\n=== Agreement Threshold Check (Subjectivity) ===')
        print('Criterion: fleiss_kappa >= 0.80')
        print(f'Computed fleiss_kappa: {kappa_value_subj:}')
        print('Threshold Check: All annotators labeled all items as subjective.')
    else:
        status_subj = 'PASS' if kappa_value_subj >= threshold else 'BELOW 0.80'
        print('\n=== Agreement Threshold Check (Subjectivity) ===')
        print('Criterion: fleiss_kappa >= 0.80')
        print(f'Computed fleiss_kappa: {kappa_value_subj:}')
        print(f'Threshold Check: {status_subj}')

=== Polarity Inter-Annotator Agreement Result ===
rows_used: 1000
annotators: 3
categories: ['POSITIVE', 'NEGATIVE']
category_proportions: {'POSITIVE': 0.428, 'NEGATIVE': 0.572}
mean_observed_agreement: 0.9666666666666668
expected_agreement: 0.5103679999999999
fleiss_kappa: 0.9319216608936237

=== Subjectivity Inter-Annotator Agreement Result ===
rows_used: 1000
annotators: 3
categories: ['SUBJECTIVE', 'NEUTRAL']
category_proportions: {'SUBJECTIVE': 1.0, 'NEUTRAL': 0.0}
mean_observed_agreement: 1.0
expected_agreement: 1.0
fleiss_kappa: nan

=== Agreement Threshold Check (Polarity) ===
Criterion: fleiss_kappa >= 0.80
Computed fleiss_kappa: 0.9319216608936237
Threshold Check: PASS

=== Agreement Threshold Check (Subjectivity) ===
Criterion: fleiss_kappa >= 0.80
Computed fleiss_kappa: nan
Threshold Check: All annotators labeled all items as subjective.


### Extra step - Verify Fleiss' kappa using statsmodels

This step independently computes Fleiss' kappa using the `statsmodels` implementation.

This provides a direct comparison to the custom implementation in the previous cell.

In [56]:
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np

# relabel polarity to numeric for Fleiss' Kappa library
numeric_matrix = combined[polarity_cols].replace({"POSITIVE": 1, "NEGATIVE": 0}).to_numpy()

flat = numeric_matrix.flatten()
flat_nonan = [x for x in flat if isinstance(x, (int, float)) and not pd.isna(x)]
category_labels = np.unique(flat_nonan)

# build counts matrix for Fleiss' Kappa
counts = np.array([
    [(row == cat).sum() for cat in category_labels]
    for row in numeric_matrix
])

fkappa = fleiss_kappa(counts)
print(f"Fleiss' Kappa (statsmodels): {fkappa}")

Fleiss' Kappa (statsmodels): 0.9319216608936237



**Analysis**

Both implementations yield nearly identical Fleiss' kappa values (0.9319), confirming the correctness of the custom calculation. 

The high kappa value indicates strong inter-annotator agreement, and the agreement threshold is clearly met.


**Conclusion**

The dataset demonstrates excellent reliability, providing a solid ground truth for training and evaluating our sentiment classification model.



## Step 5 - Create Evaluation Dataset for Model Testing

After confirming sufficient inter-annotator agreement, we use majority voting to combine the three annotators' labels into a single ground truth label for each record. 

This evaluation dataset will be used to test model performance.

In [57]:
def majority_vote(row, cols):
    labels = [row[c] for c in cols if pd.notna(row[c])]
    if not labels:
        return np.nan
    count = Counter(labels)
    return count.most_common(1)[0][0]

# Only keep rows where at least 2 annotators agree for each label
combined['ground_truth_polarity'] = combined[polarity_cols].apply(lambda row: majority_vote(row, polarity_cols), axis=1)
if subjectivity_cols:
    combined['ground_truth_subjectivity'] = combined[subjectivity_cols].apply(lambda row: majority_vote(row, subjectivity_cols), axis=1)

cols_to_export = merge_keys + []
if subjectivity_cols:
    cols_to_export.append('ground_truth_subjectivity')
cols_to_export.append('ground_truth_polarity')

eval_dataset = combined[cols_to_export].dropna(subset=['ground_truth_polarity']).copy()
print('Evaluation dataset preview:')
display(eval_dataset.head())

# save to excel for use in model evaluation
eval_dataset.to_excel('../../data/evaluation_dataset.xlsx', index=False)
print(f"Saved evaluation dataset with {len(eval_dataset)} records to '../../data/evaluation_dataset_with_subjectivity.xlsx'")

Evaluation dataset preview:


,review_id,movie_id,ground_truth_subjectivity,ground_truth_polarity
0,rw3391899,tt0478970,SUBJECTIVE,NEGATIVE
1,rw4836840,tt0478970,SUBJECTIVE,NEGATIVE
2,rw3278832,tt0478970,SUBJECTIVE,NEGATIVE
3,rw4581027,tt5095030,SUBJECTIVE,NEGATIVE
4,rw4372367,tt5095030,SUBJECTIVE,NEGATIVE


Saved evaluation dataset with 1000 records to '../../data/evaluation_dataset_with_subjectivity.xlsx'
